In [1]:
import pandas as pd
import numpy as np
import re
from typing import NamedTuple

## Data

In [2]:
df_wine = pd.read_csv('wine-knowledge-discovery/data/WineDataset.csv')

In [3]:
df_wine.head()

,Title,Description,Price,Capacity,Grape,Secondary Grape Varieties,Closure,Country,Unit,Characteristics,Per bottle / case / each,Type,ABV,Region,Style,Vintage,Appellation
0,"The Guv'nor, Spain",We asked some of our most prized winemakers wo...,£9.99 per bottle,75CL,Tempranillo,NaN,Natural Cork,Spain,10.5,"Vanilla, Blackberry, Blackcurrant",per bottle,Red,ABV 14.00%,NaN,Rich & Juicy,NV,NaN
1,Bread & Butter 'Winemaker's Selection' Chardon...,This really does what it says on the tin. It’s...,£15.99 per bottle,75CL,Chardonnay,NaN,Natural Cork,USA,10.1,"Vanilla, Almond, Coconut, Green Apple, Peach, ...",per bottle,White,ABV 13.50%,California,Rich & Toasty,2021,Napa Valley
2,"Oyster Bay Sauvignon Blanc 2022, Marlborough",Oyster Bay has been an award-winning gold-stan...,£12.49 per bottle,75CL,Sauvignon Blanc,NaN,Screwcap,New Zealand,9.8,"Tropical Fruit, Gooseberry, Grapefruit, Grass,...",per bottle,White,ABV 13.00%,Marlborough,Crisp & Zesty,2022,NaN
3,Louis Latour Mâcon-Lugny 2021/22,We’ve sold this wine for thirty years – and fo...,£17.99 per bottle,75CL,Chardonnay,NaN,Natural Cork,France,10.1,"Peach, Apricot, Floral, Lemon",per bottle,White,ABV 13.50%,Burgundy,Ripe & Rounded,2022,Macon
4,Bread & Butter 'Winemaker's Selection' Pinot N...,Bread & Butter is that thing that you can coun...,£15.99 per bottle,75CL,Pinot Noir,NaN,Natural Cork,USA,10.1,"Smoke, Black Cherry, Cedar, Raspberry, Red Fruit",per bottle,Red,ABV 13.50%,California,Smooth & Mellow,2021,Napa Valley


### volumes Liters fix (standartisation)

In [4]:
df_wine.Capacity.value_counts()

Capacity
75CL      1193
37.5CL      23
750ML       18
1.5LTR      18
150CL       11
50CL         8
Our          6
2.25L        4
70CL         3
500ML        3
300CL        1
5LITRE       1
375ML        1
Name: count, dtype: int64

In [4]:
def convert_to_liters(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).upper().strip()
    
    # Extract numeric part
    number = re.findall(r"[\d.]+", value)
    if not number:
        return np.nan
    
    number = float(number[0])
    
    # Convert based on unit
    if "ML" in value:
        return number / 1000
    elif "CL" in value:
        return number / 100
    elif "L" in value:   # catches L, LTR, LITRE
        return number
    else:
        return np.nan

# Apply
df_wine["Capacity_L"] = df_wine["Capacity"].apply(convert_to_liters)

### Characteristics OHE

In [5]:
ohe = df_wine["Characteristics"].str.get_dummies(sep=", ")
ohe.head()

,Almond,Apricot,Asparagus,Banana,Biscuit,Black Cherry,Black Fruit,Black Pepper,Black Plum,Blackberry,...,Tobacco,Toffee,Tomato Leaf,Tropical Fruit,Vanilla,Violet,Walnut,Watermelon,Wet Stones,White Pepper
0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Style OHE

In [9]:
df_wine['Style'].unique()

array(['Rich & Juicy', 'Rich & Toasty', 'Crisp & Zesty', 'Ripe & Rounded',
       'Smooth & Mellow', nan, 'Light & Refreshing', 'Crisp & Fruity',
       'Fresh & Elegant ', 'Delicate & Dry', 'Bold & Spicy',
       'Savoury & Full Bodied', 'Soft & Fruity', 'Aromatic & Floral',
       'Smooth & Light', 'Ripe & Fruity', 'Light & Elegant'], dtype=object)

In [6]:
df_wine["Style"] = (
    df_wine["Style"]
    .str.strip()                 # remove trailing spaces
)

# Split on &
split_styles = (
    df_wine["Style"]
    .dropna()
    .str.split("&")
    .apply(lambda x: [i.strip() for i in x])
)

# One-hot encode
ohe2 = (
    split_styles
    .explode()
    .str.get_dummies()
    .groupby(level=0)
    .sum()
)

In [13]:
ohe2

,Aromatic,Bold,Crisp,Delicate,Dry,Elegant,Floral,Fresh,Fruity,Full Bodied,...,Refreshing,Rich,Ripe,Rounded,Savoury,Smooth,Soft,Spicy,Toasty,Zesty
0,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
2,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
3,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1285,1,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1286,0,0,0,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1287,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0
1288,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,0,0,0


In [18]:
%pip install mlxtend

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.3 MB/s eta 0:00:0000:0100:01
  Using cached matplotlib-3.10.8-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.7 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.2 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [11]:
ohe = ohe.add_prefix("ch_")
ohe2 = ohe2.add_prefix("st_")

In [12]:
ohe3 = df_wine["Style"].str.get_dummies()

In [16]:
df_wine.Grape.unique()

array(['Tempranillo', 'Chardonnay', 'Sauvignon Blanc', 'Pinot Noir',
       'Glera', nan, 'Chenin Blanc', 'Castelão ', 'Malagousia',
       'Cinsault', 'Grenache', 'Shiraz', 'Cabernet Sauvignon', 'Bacchus',
       'Viognier', 'Pinot Grigio', 'Garnacha', 'Malbec', 'Cortese',
       'Merlot', 'Melon De Bourgogne', 'Carménère', 'Zinfandel', 'Syrah',
       'Marsanne', 'Gruner Veltliner', 'Corvina', 'Greco', 'Macabeo',
       'Gamay', 'Loureiro', 'Riesling', 'Alvarinho', 'Mourvèdre',
       'Cabernet Franc', 'Vespaiola', 'Picpoul', 'Vermentino',
       'Sangiovese', 'Pinot Meunier', 'Verdejo', 'Primitivo', 'Pinotage',
       'Alicante Bouschet', 'Garganega', 'Godello', 'Carignan',
       'Grenache Blanc', 'Aligoté', 'Siegerrebe', 'Touriga Nacional',
       'Albarino', 'Nerello Mascalese', "Nero D'Avola", 'Turbiana',
       'Pinot Gris', 'Airen', 'Trincadeira', 'Tinta Roriz', 'Xinomavro',
       'Agiorgitiko', 'Pais', 'Gewürztraminer', 'Mencia', 'Verdicchio',
       'Fiano', 'Rondinella', '

In [7]:
ohe_grape = df_wine["Grape"].str.get_dummies()

In [15]:
char_cols = list(ohe.columns)

In [34]:
styl_cols = list(ohe2.columns)
styl_cols2 = list(ohe3.columns)

In [35]:
ohe_cs = ohe.join(ohe3)

In [17]:
df_wine = df_wine.join(ohe)

In [18]:
df_wine = df_wine.join(ohe2)

In [17]:
context2 = ohe.join(ohe_grape)

In [ ]:
# fca_df = df_wine[char_cols + styl_cols].astype(bool)

fca_df = ohe_cs.astype(bool)
fca_df2 = context2.astype(bool)


In [38]:
fca_df.head()

,ch_Almond,ch_Apricot,ch_Asparagus,ch_Banana,ch_Biscuit,ch_Black Cherry,ch_Black Fruit,ch_Black Pepper,ch_Black Plum,ch_Blackberry,...,Light & Elegant,Light & Refreshing,Rich & Juicy,Rich & Toasty,Ripe & Fruity,Ripe & Rounded,Savoury & Full Bodied,Smooth & Light,Smooth & Mellow,Soft & Fruity
0,False,False,False,False,False,False,False,False,False,True,...,False,False,True,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,True,False,False,False,False
4,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False


In [19]:
fca_df2.head()

,ch_Almond,ch_Apricot,ch_Asparagus,ch_Banana,ch_Biscuit,ch_Black Cherry,ch_Black Fruit,ch_Black Pepper,ch_Black Plum,ch_Blackberry,...,Verdejo,Verdicchio,Vermentino,Vespaiola,Viognier,Viura,Xarel-Lo,Xinomavro,Zinfandel,Zwieigelt
0,False,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,True,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [20]:
from mlxtend.frequent_patterns import apriori, association_rules

In [44]:


# freq = apriori(fca_df, min_support=0.005, use_colnames=True, low_memory=True)
freq2 = apriori(fca_df2, min_support=0.01, use_colnames=True, low_memory=True)



In [30]:
# rules = association_rules(freq, metric="confidence", min_threshold=0.6)
rules2 = association_rules(freq2, metric="confidence", min_threshold=0.6)

In [45]:
freq2

,support,itemsets
0,0.034109,(ch_Almond)
1,0.068992,(ch_Apricot)
2,0.031783,(ch_Biscuit)
3,0.096124,(ch_Black Cherry)
4,0.116279,(ch_Black Fruit)
...,...,...
744,0.010853,"(ch_Raspberry, Grenache, ch_Peach, ch_Strawber..."
745,0.010078,"(ch_Black Plum, ch_Blackcurrant, ch_Tobacco, c..."
746,0.011628,"(ch_Black Plum, ch_Raspberry, Grenache, ch_Spi..."
747,0.011628,"(ch_Lemon, ch_Peach, ch_Green Apple, ch_Vanill..."


In [93]:
freq

,support,itemsets
0,0.034109,(ch_Almond)
1,0.068992,(ch_Apricot)
2,0.031783,(ch_Biscuit)
3,0.096124,(ch_Black Cherry)
4,0.116279,(ch_Black Fruit)
...,...,...
1888,0.006202,"(ch_Blueberry, ch_Black Plum, ch_Tobacco, ch_C..."
1889,0.006202,"(ch_Blackcurrant, ch_Blueberry, ch_Black Plum,..."
1890,0.006202,"(ch_Blackcurrant, ch_Blueberry, ch_Tobacco, ch..."
1891,0.006202,"(ch_Stone Fruit, ch_Butter, ch_Hazelnut, ch_Br..."


In [94]:
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(ch_Apricot),(ch_Peach),0.068992,0.189922,0.055039,0.797753,4.200413,1.0,0.041936,4.005383,0.818391,0.269962,0.750336,0.543774
1,(ch_Biscuit),(ch_Bread),0.031783,0.065116,0.024806,0.780488,11.986063,1.0,0.022737,4.258915,0.946657,0.344086,0.765198,0.580720
2,(ch_Biscuit),(ch_Citrus Fruit),0.031783,0.185271,0.023256,0.731707,3.949383,1.0,0.017367,3.036716,0.771310,0.120000,0.670697,0.428615
3,(ch_Biscuit),(Rich & Toasty),0.031783,0.109302,0.022481,0.707317,6.471199,1.0,0.019007,3.043217,0.873223,0.189542,0.671400,0.456495
4,(ch_Ripe Fruit),(ch_Black Fruit),0.029457,0.116279,0.024806,0.842105,7.242105,1.0,0.021381,5.596899,0.888079,0.205128,0.821330,0.527719
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6150,"(Bold & Spicy, ch_Chocolate, ch_Tobacco)","(ch_Blackcurrant, ch_Blueberry, ch_Black Plum,...",0.006977,0.008527,0.006202,0.888889,104.242424,1.0,0.006142,8.923256,0.997365,0.666667,0.887933,0.808081
6151,"(ch_Blackberry, Bold & Spicy, ch_Tobacco)","(ch_Blackcurrant, ch_Blueberry, ch_Black Plum,...",0.007752,0.009302,0.006202,0.800000,86.000000,1.0,0.006129,4.953488,0.996094,0.571429,0.798122,0.733333
6152,"(Bold & Spicy, ch_Vanilla, ch_Tobacco)","(ch_Blackcurrant, ch_Blueberry, ch_Black Plum,...",0.007752,0.010853,0.006202,0.800000,73.714286,1.0,0.006117,4.945736,0.994141,0.500000,0.797806,0.685714
6153,"(ch_Blueberry, ch_Tobacco)","(ch_Blackcurrant, ch_Black Plum, ch_Chocolate,...",0.010078,0.010078,0.006202,0.615385,61.065089,1.0,0.006100,2.573798,0.993637,0.444444,0.611469,0.615385


In [95]:
rules_df = rules[
    rules['antecedents'].apply(lambda x: any(i in styl_cols2 for i in x)) &
    rules['consequents'].apply(lambda x: any(i in char_cols for i in x))
]

In [46]:
rules_df2 = rules2[
    rules2['antecedents'].apply(lambda x: any(i in ohe_grape.columns for i in x)) &
    rules2['consequents'].apply(lambda x: any(i in ohe.columns for i in x))
]

In [47]:
rules_df.head(50)

NameError: name 'rules_df' is not defined

In [48]:
rules_df2.head(50)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
1,(Viognier),(ch_Apricot),0.016279,0.068992,0.013178,0.809524,11.733547,1.0,0.012055,4.887791,0.929912,0.182796,0.795409,0.500268
6,(Malbec),(ch_Black Fruit),0.028682,0.116279,0.022481,0.783784,6.740541,1.0,0.019145,4.087209,0.876792,0.183544,0.755334,0.488559
11,(Shiraz),(ch_Black Plum),0.033333,0.127907,0.023256,0.697674,5.454545,1.0,0.018992,2.884615,0.844828,0.168539,0.653333,0.439746
14,(Shiraz),(ch_Blackberry),0.033333,0.124031,0.024806,0.744186,6.000000,1.0,0.020672,3.424242,0.862069,0.187135,0.707965,0.472093
15,(Syrah),(ch_Blackberry),0.013953,0.124031,0.009302,0.666667,5.375000,1.0,0.007572,2.627907,0.825472,0.072289,0.619469,0.370833
18,(Cabernet Sauvignon),(ch_Blackcurrant),0.067442,0.111628,0.047287,0.701149,6.281130,1.0,0.039758,2.972630,0.901598,0.358824,0.663598,0.562380
23,(Glera),(ch_Citrus Fruit),0.013953,0.185271,0.009302,0.666667,3.598326,1.0,0.006717,2.444186,0.732311,0.048980,0.590866,0.358438
29,(Sauvignon Blanc),(ch_Grass),0.079070,0.060465,0.054264,0.686275,11.349925,1.0,0.049483,2.994767,0.990188,0.636364,0.666084,0.791855
31,(Sauvignon Blanc),(ch_Green Apple),0.079070,0.189147,0.058140,0.735294,3.887416,1.0,0.043184,3.063221,0.806532,0.276753,0.673546,0.521336
34,(Viognier),(ch_Peach),0.016279,0.189922,0.013178,0.809524,4.262391,1.0,0.010087,4.252907,0.778056,0.068273,0.764867,0.439456


In [49]:
rules_df2_2 = rules2[
    rules2['consequents'].apply(lambda x: any(i in ohe_grape.columns for i in x)) &
    rules2['antecedents'].apply(lambda x: any(i in ohe.columns for i in x))
    
]

In [51]:
rules_df2_2.head(50)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
4,(ch_Biscuit),(Chardonnay),0.031783,0.183721,0.026357,0.829268,4.513739,1.0,0.020517,4.781063,0.804008,0.139344,0.790841,0.486364
19,(ch_Bread),(Chardonnay),0.065116,0.183721,0.050388,0.773810,4.211875,1.0,0.038424,3.608813,0.815691,0.253906,0.722901,0.524036
22,(ch_Butter),(Chardonnay),0.034884,0.183721,0.031008,0.888889,4.838256,1.0,0.024599,7.346512,0.821988,0.165289,0.863881,0.528833
24,(ch_Cream),(Chardonnay),0.059690,0.183721,0.041085,0.688312,3.746507,1.0,0.030119,2.618895,0.779620,0.203065,0.618160,0.455970
26,(ch_Gooseberry),(Sauvignon Blanc),0.023256,0.079070,0.020930,0.900000,11.382353,1.0,0.019091,9.209302,0.933862,0.257143,0.891414,0.582353
30,(ch_Grass),(Sauvignon Blanc),0.060465,0.079070,0.054264,0.897436,11.349925,1.0,0.049483,8.979070,0.970580,0.636364,0.888630,0.791855
32,(ch_Hazelnut),(Chardonnay),0.027132,0.183721,0.020155,0.742857,4.043400,1.0,0.015170,3.174419,0.773675,0.105691,0.684982,0.426281
33,(ch_Pastry),(Chardonnay),0.013178,0.183721,0.009302,0.705882,3.842144,1.0,0.006881,2.775349,0.749607,0.049587,0.639685,0.378258
44,(ch_Ripe Fruit),(Malbec),0.029457,0.028682,0.020930,0.710526,24.772404,1.0,0.020085,3.355462,0.988759,0.562500,0.701978,0.720128
53,"(ch_Apricot, ch_Lemon)",(Chardonnay),0.026357,0.183721,0.018605,0.705882,3.842144,1.0,0.013762,2.775349,0.759753,0.097166,0.639685,0.403574


In [98]:
%pip install concepts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 KB 2.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [99]:
import concepts

In [105]:
context = pd.concat(
    [ohe, ohe3],
    axis=1
).astype(bool)
fca_df_l = context.replace({True: "X", False: ""})

objects = fca_df_l.index.tolist()
attributes = fca_df_l.columns.tolist()
relation = fca_df_l.values.tolist()

# 5. Create FCA context
c = concepts.Context(objects, attributes, relation)

# c = concepts.Context.from_pandas(fca_df_l)

In [106]:
for extent, intent in c:
    print("Objects:", extent)
    print("Attributes:", intent)
    print()

TypeError: 'int' object is not iterable

In [ ]:
style_cols = set(ohe3.columns)
chare_cols = set(ohe.columns)

rules = []

for premise, conclusion in c.lattice.implications:
    if set(premise).issubset(style_cols):
        char_links = set(conclusion).intersection(chare_cols)
        if char_links:
            rules.append((premise, char_links))

rules


#### Caspi / Paspi

In [13]:
import caspailleur as csp

In [19]:
ohe2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1212 entries, 0 to 1289
Data columns (total 23 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Aromatic     1212 non-null   int64
 1   Bold         1212 non-null   int64
 2   Crisp        1212 non-null   int64
 3   Delicate     1212 non-null   int64
 4   Dry          1212 non-null   int64
 5   Elegant      1212 non-null   int64
 6   Floral       1212 non-null   int64
 7   Fresh        1212 non-null   int64
 8   Fruity       1212 non-null   int64
 9   Full Bodied  1212 non-null   int64
 10  Juicy        1212 non-null   int64
 11  Light        1212 non-null   int64
 12  Mellow       1212 non-null   int64
 13  Refreshing   1212 non-null   int64
 14  Rich         1212 non-null   int64
 15  Ripe         1212 non-null   int64
 16  Rounded      1212 non-null   int64
 17  Savoury      1212 non-null   int64
 18  Smooth       1212 non-null   int64
 19  Soft         1212 non-null   int64
 20  Spicy        

In [26]:
PairType = pd.concat([ohe_grape, ohe2], axis=1)


In [27]:

concepts_df = csp.mine_concepts(PairType.astype(bool))

print(concepts_df[['extent', 'intent']].map(', '.join))

                                                       extent  \
concept_id                                                      
0           382, 564, 876, 644, 204, 905, 1062, 771, 331, ...   
1           564, 1069, 1071, 445, 1043, 289, 1189, 317, 89...   
2           270, 285, 1164, 1151, 445, 644, 153, 1043, 188...   
3           816, 709, 217, 445, 1043, 456, 862, 905, 824, ...   
4           255, 710, 1069, 356, 1071, 153, 1189, 824, 16,...   
...                                                       ...   
363                                                       667   
364                                                      1191   
365                                                       875   
366                                                       966   
367                                                             

                                                       intent  
concept_id                                                     
0                         

In [28]:
implications_df = csp.mine_implications(
  PairType.astype(bool), basis_name='Canonical', unit_base=True,
  to_compute=['premise', 'conclusion', 'extent'],
  min_support=2,
)

implications_df

,premise,conclusion,extent
implication_id,,,
0,{Tinta Barroca},Aromatic,"{954, 1250, 480}"
1,{Tinta Barroca},Bold,"{954, 1250, 480}"
2,{Tinta Barroca},Crisp,"{954, 1250, 480}"
3,{Tinta Barroca},Delicate,"{954, 1250, 480}"
4,{Tinta Barroca},Dry,"{954, 1250, 480}"
...,...,...,...
2614,"{Delicate, Dry, Savoury, Full Bodied}",Smooth,"{445, 1043, 893, 970, 868, 1191, 510, 482, 631..."
2615,"{Delicate, Dry, Savoury, Full Bodied}",Soft,"{445, 1043, 893, 970, 868, 1191, 510, 482, 631..."
2616,"{Delicate, Dry, Savoury, Full Bodied}",Spicy,"{445, 1043, 893, 970, 868, 1191, 510, 482, 631..."


In [34]:
PairType.to_csv("char_grape.csv", index=True)

In [32]:
concepts_df = csp.mine_concepts(PairType.astype(bool), min_support=5)

# manually define what to show in the nodes of the diagram
new_intent_labels = ('<b>' + concepts_df['new_intent'].map(sorted).map(', '.join) + '</b>').replace('<b></b>', '')
old_intent_labels = (concepts_df['intent'] - concepts_df['new_intent']).map(sorted).map(', '.join)
intent_labels = (new_intent_labels + ';' + old_intent_labels).map(lambda l: ', '.join(l.strip(';').split(';')))
extent_labels = concepts_df['extent'].map(sorted).map(', '.join)

node_labels = intent_labels + '<br><br>' + extent_labels
node_labels = [l.replace(' ', '&nbsp') for l in node_labels] # replace space with non-breakable space for better Mermaid visualisation

diagram_code = csp.io.to_mermaid_diagram(node_labels, concepts_df['previous_concepts'])
print(diagram_code)

flowchart TD
A["<br><br>0,&nbsp1,&nbsp10,&nbsp100,&nbsp1000,&nbsp1001,&nbsp1002,&nbsp1003,&nbsp1004,&nbsp1005,&nbsp1006,&nbsp1007,&nbsp1008,&nbsp1009,&nbsp101,&nbsp1010,&nbsp1011,&nbsp1012,&nbsp1013,&nbsp1014,&nbsp1015,&nbsp1016,&nbsp1017,&nbsp1018,&nbsp1019,&nbsp102,&nbsp1020,&nbsp1021,&nbsp1022,&nbsp1023,&nbsp1024,&nbsp1025,&nbsp1026,&nbsp1027,&nbsp1028,&nbsp1029,&nbsp103,&nbsp1030,&nbsp1031,&nbsp1032,&nbsp1033,&nbsp1034,&nbsp1035,&nbsp1036,&nbsp1037,&nbsp1038,&nbsp1039,&nbsp104,&nbsp1040,&nbsp1041,&nbsp1042,&nbsp1043,&nbsp1044,&nbsp1045,&nbsp1046,&nbsp1047,&nbsp1048,&nbsp1049,&nbsp105,&nbsp1050,&nbsp1051,&nbsp1052,&nbsp1053,&nbsp1054,&nbsp1055,&nbsp1056,&nbsp1057,&nbsp1058,&nbsp1059,&nbsp106,&nbsp1060,&nbsp1061,&nbsp1062,&nbsp1063,&nbsp1064,&nbsp1065,&nbsp1066,&nbsp1067,&nbsp1068,&nbsp1069,&nbsp107,&nbsp1070,&nbsp1071,&nbsp1072,&nbsp1073,&nbsp1074,&nbsp1075,&nbsp1076,&nbsp1077,&nbsp1078,&nbsp1079,&nbsp108,&nbsp1080,&nbsp1081,&nbsp1082,&nbsp1083,&nbsp1084,&nbsp1085,&nbsp1086,&nbsp108

### Regions

In [42]:
df_wine[df_wine['Grape'] == 'Shiraz']

,Title,Description,Price,Capacity,Grape,Secondary Grape Varieties,Closure,Country,Unit,Characteristics,Per bottle / case / each,Type,ABV,Region,Style,Vintage,Appellation,Capacity_L
19,"Two Hands 'Angels' Share' Shiraz 2021/22, McLa...",Two Hands successfully prove that not all Auss...,£21.99 per bottle,75CL,Shiraz,NaN,Screwcap,Australia,10.8,"Vanilla, Black Fruit, Black Pepper, Blackcurra...",per bottle,Red,ABV 14.40%,South Australia,Bold & Spicy,2022,Mclaren Vale,0.75
54,"Two Hands 'Tenacity' Old Vine Shiraz 2021/22, ...",Two Hands founders Michael Twelftree and Richa...,£14.99 per bottle,75CL,Shiraz,NaN,Natural Cork,Australia,10.7,"Vanilla, Black Pepper, Black Plum, Blackberry,...",per bottle,Red,ABV 14.20%,South Australia,Bold & Spicy,2022,Mclaren Vale,0.75
70,"Jim Barry 'The Lodge Hill' Shiraz 2020/21, Cla...",Clare Valley has a relatively cool climate. It...,£15.99 per bottle,75CL,Shiraz,NaN,Screwcap,Australia,10.5,"Vanilla, Black Plum, Blackberry, Blackcurrant,...",per bottle,Red,ABV 14.00%,South Australia,Bold & Spicy,2021,Clare Valley,0.75
78,"Peter Lehmann 'The Barossan' Shiraz 2019/20, B...","With rolling hills, hot, dry summers and some ...",£18.99 per bottle,75CL,Shiraz,NaN,Screwcap,Australia,10.9,"Smoke, Black Pepper, Black Plum, Blackberry, B...",per bottle,Red,ABV 14.50%,South Australia,Bold & Spicy,2020,Barossa Valley,0.75
163,"Casella 'PepperBox' Shiraz 2020/21, South East...","In 1820, in the small Sicilian town of Giarre,...",£11.99 per bottle,75CL,Shiraz,NaN,Screwcap,Australia,10.5,"Black Pepper, Black Plum, Spice, Vanilla",per bottle,Red,ABV 14.00%,South East Australia,Bold & Spicy,2021,NaN,0.75
172,"Boekenhoutskloof ‘The Wolftrap’ Shiraz 2021, S...","Love big, bold reds? Then you’ve likely heard ...",£9.99 per bottle,75CL,Shiraz,NaN,Screwcap,South Africa,10.9,"Black Fruit, Black Pepper, Black Plum, Blackbe...",per bottle,Red,ABV 14.50%,Western Cape,Bold & Spicy,2021,NaN,0.75
210,"Wynns Coonawarra Estate Shiraz 2020/21, Coonaw...",Coonawarra's cool climate and famous terra ros...,£14.99 per bottle,75CL,Shiraz,NaN,Screwcap,Australia,10.8,"Black Plum, Blackberry, Blackcurrant, Blueberr...",per bottle,Red,ABV 14.40%,South East Australia,Bold & Spicy,2021,Coonawarra,0.75
238,Penfolds 'Koonunga Hill Seventy Six' Shiraz-Ca...,"When it comes to reds from Down Under, you can...",£16.99 per bottle,75CL,Shiraz,Cabernet Sauvignon,Screwcap,Australia,10.9,"Black Plum, Blackberry, Blackcurrant, Blueberr...",per bottle,Red,ABV 14.50%,South Australia,Bold & Spicy,2021,NaN,0.75
268,"Jam Shed Shiraz 2021/22, Australia",Jam Shed was created with one thing in mind – ...,£8.99 per bottle,75CL,Shiraz,NaN,Screwcap,Australia,10.1,"Jammy, Red Cherry, Red Plum, Vanilla",per bottle,Red,ABV 13.50%,NaN,Rich & Juicy,2022,NaN,0.75
361,"Penfolds 'Kalimna Bin 28' Shiraz 2019/20, Aust...",Kalimna Bin 28 was first produced in 1959 and ...,£34.99 per bottle,75CL,Shiraz,NaN,Natural Cork,Australia,10.9,"Vanilla, Black Pepper, Black Plum, Blackberry,...",per bottle,Red,ABV 14.50%,South Australia,Bold & Spicy,2020,Barossa Valley,0.75


In [11]:
df_wine.Region.value_counts()

Region
Burgundy           101
Bordeaux            67
Marlborough         65
Loire               59
South Australia     56
                  ... 
Oregon               1
Coastal              1
Elgin                1
Bulgaria             1
Basilicata           1
Name: count, Length: 94, dtype: int64

## adapt

In [ ]:
# Here are some of possible binary attributes. You can always propose your own attributes.
titanic_bin = {
    'Survived': titanic_df['Survived'] == 1,
    'FirstClass': titanic_df['Pclass'] == 1,
    'French': titanic_df['Embarked'] == 'C',  # 'C' means 'Cherbourg'
    'Adult': titanic_df['Age'] >= 18,
    'Child': titanic_df['Age'] < 18,
    'Male': titanic_df['Sex'] == 'male',
    'Female': titanic_df['Sex'] == 'female',
}
titanic_bin = pd.DataFrame(titanic_bin)
titanic_bin.head()